# NF3 · Integridad y calidad de datos (RA3)
## Auditando los reportes anti-cheat · *juego online*

Antes de actuar sobre un reporte anti-cheat hay que **confiar** en él. Tu misión: auditar la integridad y la calidad de los reportes.

Tres frentes:
- **A) Integridad distribuida (RA3.3/3.4):** marcador `_SUCCESS` y **sumas de
  verificación** (checksums).
- **B) Baseline anti-enmascaramiento (RA3.1):** la media/desviación típica
  *enmascara* anomalías; la mediana/MAD las detecta.
- **C) Calidad como código con dbt (RA3.1/3.2):** `dbt test` sobre los reportes.

> Autoevaluable: la celda final genera `resultados.json`. Toda la evidencia es
> fichero/dato (sin capturas).

### Antes de empezar · genera los datos (una sola vez)

Abre una **terminal** (no una celda) y ejecuta, **desde la raíz del repositorio**.
**El orden importa:** `preparar_integridad.py` trabaja sobre los datos ya generados.

```bash
cd nf3
python datos/generar_datos.py --salida datos/raw
python datos/preparar_integridad.py
```

Este cuaderno **se sitúa solo** en `nf3/`, así que todas sus rutas son relativas a esa
carpeta. Si la primera celda de código falla con un error de fichero no encontrado, es
que te falta este paso.


In [ ]:
import os
if os.path.basename(os.getcwd()) == "actividad": os.chdir("..")  # ejecutar desde la carpeta del núcleo (donde está datos/)
import hashlib, json, os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
DIST = Path("almacen/alertas_dist"); INCOMP = Path("almacen/alertas_incompletas")
DBT = Path("dbt_calidad")
resultados = {}
print("Listo.")

---
## A) Integridad distribuida (RA3.3/3.4)
1. Comprueba el marcador **`_SUCCESS`** en `almacen/alertas_dist` (job completado).
2. Comprueba que `almacen/alertas_incompletas` **no** lo tiene (job incompleto → no fiar).
3. Recalcula la **suma de verificación** (sha256) de cada part-file y compárala con
   `manifiesto.json`: detecta cuántos ficheros están **corruptos**.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for ch in iter(lambda: f.read(8192), b""):
            h.update(ch)
    return h.hexdigest()

# A · Integridad — completa las 3 comprobaciones
success = ...        # TODO: ¿existe el marcador _SUCCESS en DIST? · pista: (DIST/"_SUCCESS").exists()
incompleto = ...     # TODO: ¿le FALTA el _SUCCESS al directorio INCOMP? · pista: not (INCOMP/"_SUCCESS").exists()
manifiesto = json.load(open(DIST/"manifiesto.json"))
corruptos = ...      # TODO: ficheros cuyo sha256 actual NO coincide con el manifiesto
                     #       pista: [f for f,chk in manifiesto.items() if sha256(DIST/f) != chk]
resultados["integridad"] = {
    "success_marker_presente": bool(success),
    "directorio_incompleto_detectado": bool(incompleto),
    "n_ficheros_verificados": len(manifiesto),
    "n_ficheros_corruptos": len(corruptos),
    "integridad_global_ok": bool(len(corruptos) == 0 and success),
}
resultados["integridad"]

---
## B) Baseline anti-enmascaramiento (RA3.1)
En `datos/raw/lecturas.csv` hay una serie de medidas de un jugador con **picos extremos**.
Compara dos formas de fijar el umbral de anomalía:
- **No robusto:** media + 3·desviación típica (los picos inflan la std).
- **Robusto:** mediana + 3·(1.4826·MAD).

Cuenta cuántas anomalías detecta cada método. El robusto debería detectar **más**
(porque el no robusto las *enmascara*).

In [ ]:
x = pd.read_csv("datos/raw/lecturas.csv")["lectura"].values
# B · Baseline anti-enmascaramiento — completa los dos métodos
umbral_media = ...      # TODO: umbral NO robusto · pista: media + 3*desviación típica
anom_media = ...        # TODO: nº de lecturas por encima de ese umbral · pista: (x > umbral).sum()
med = np.median(x); mad = np.median(np.abs(x - med))
umbral_robusto = ...    # TODO: umbral ROBUSTO · pista: mediana + 3*(1.4826*MAD)
anom_robusto = ...      # TODO: nº de lecturas por encima del umbral robusto
resultados["baseline"] = {
    "anomalias_metodo_media": int(anom_media),
    "anomalias_metodo_robusto": int(anom_robusto),
    "enmascaramiento_evitado": bool(anom_robusto > anom_media),
}
resultados["baseline"]

---
## C) Calidad como código con dbt (RA3.1/3.2)
Abre `dbt_calidad/models/schema.yml` y **completa los tests** de `severidad`
(`not_null` y `accepted_values` con baja/media/alta/critica). Crea el test singular
`dbt_calidad/tests/puntuacion_sospecha_en_rango.sql` (filas con `puntuacion_sospecha < 0`
o `> 500`).

Luego esta celda ejecuta `dbt test` y resume los resultados.
> **Nota.** Los cuatro tests que usas aquí (`unique`, `not_null`, `accepted_values` y el
> singular) son los que trae dbt de serie. Existen paquetes con **aserciones** más ricas,
> como `dbt-expectations`, pero **no están instalados en este entorno** y no hacen falta
> para la actividad: no intentes usarlos.

In [ ]:
# C · Ejecuta dbt EN PROCESO (dbtRunner) y resume los resultados
from dbt.cli.main import dbtRunner
_cwd = os.getcwd(); os.chdir(DBT)   # dbt-duckdb resuelve read_parquet relativo al proyecto
try:
    r = dbtRunner()
    r.invoke(["run", "--quiet"]); r.invoke(["test", "--quiet"])
    rr = json.load(open("target/run_results.json"))
    mani = json.load(open("target/manifest.json"))
finally:
    os.chdir(_cwd)
def etiqueta(uid):
    n = mani["nodes"].get(uid, {}); tm = n.get("test_metadata")
    if tm:
        return f'{tm["name"]}:{tm.get("kwargs",{}).get("column_name","")}'.strip(":")
    return uid.split(".")[-1]
fallos = {etiqueta(x["unique_id"]): x.get("failures",0) for x in rr["results"] if x["status"]=="fail"}
resultados["calidad_dbt"] = {"n_tests": len(rr["results"]),
    "n_pass": sum(1 for x in rr["results"] if x["status"]=="pass"),
    "n_fail": sum(1 for x in rr["results"] if x["status"]=="fail"),
    "fallos_por_test": dict(sorted(fallos.items()))}
resultados["calidad_dbt"]

---
## D) Razonamiento (formato examen, RA3)
En **máximo 12 líneas**, con terminología precisa:

**a)** Explica cómo se calcula una **línea base** con la **mediana** y por qué eso
evita el **enmascaramiento estadístico** que produciría la media (parte B).

**b)** ¿Qué función cumple el marcador **`_SUCCESS`** en un sistema de ficheros
distribuido y qué debes **inferir** ante su ausencia? (RA3.4)

**c)** ¿Por qué a mayor volumen de datos crece el **riesgo de integridad** y qué
papel juega la **suma de verificación**? (RA3.2/3.3)

*(Escribe aquí tu respuesta.)*


---
## Celda final · Generar `resultados.json` (no modificar)

In [ ]:
ALUMNO = "TU_NOMBRE_Y_APELLIDOS"   # <-- pon aquí "Apellidos, Nombre"
resultados["metadata"] = {"caso":"gaming","seed":42,"alumno":ALUMNO}
assert ALUMNO != "TU_NOMBRE_Y_APELLIDOS", "⚠️ Pon tus Apellidos, Nombre en ALUMNO antes de entregar."
assert set(resultados) >= {"integridad","baseline","calidad_dbt"}, "Faltan secciones"
json.dump(resultados, open("resultados.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("✅ resultados.json generado. Entrega el .ipynb y la carpeta dbt_calidad.")

---
### Checkpoint de preparación al examen (no evaluable)
1. Un jugador tiene 3 picos enormes y 5 valores moderadamente altos: ¿qué método de
   umbral los detecta todos y por qué?
2. Encuentras un directorio de resultados sin `_SUCCESS`: ¿lo usas? ¿por qué?
3. ¿Qué garantiza una suma de verificación que no garantiza el simple tamaño del fichero?